## Google Colab GPU Setup
To enable GPU acceleration in Google Colab, please follow these steps:
1. Go to the **Runtime** menu.
2. Select **Change runtime type**.
3. Under **Hardware accelerator**, select **GPU** (e.g., T4, L4, or A100).
4. Click **Save**.

In [ ]:
# Google Colab Setup
# To enable GPU: Runtime -> Change runtime type -> Hardware accelerator -> GPU
import os
import sys

try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

if IN_COLAB:
    print("Detected Google Colab. Installing dependencies...")
    !pip install -q warp-lang pydantic pyyaml scipy matplotlib
    if not os.path.exists('pazuzu'):
        !git clone https://github.com/dzilles/pazuzu.git
    
    repo_path = os.path.abspath('pazuzu')
    if repo_path not in sys.path:
        sys.path.append(repo_path)
    print(f"Project root added to sys.path: {repo_path}")
else:
    print("Running locally. Skipping Colab setup.")

# Isentropic Vortex Verification

This notebook validates the High-Order Flux Reconstruction solver using the Isentropic Vortex test case.

## Euler Equations
We solve the 2D Compressible Euler equations:
$$\frac{\partial \mathbf{q}}{\partial t} + \nabla \cdot \mathbf{F}(\mathbf{q}) = 0$$
where $\mathbf{q} = [\rho, \rho u, \rho v, E]^T$.

## Exact Solution
The isentropic vortex is an exact solution of the Euler equations representing a perturbation convecting in a freestream.
The perturbations are given by (where $r$ is the normalized radius $r_{phys}/R$):
$$ \delta u = -\frac{S}{2\pi} \frac{y-y_c}{R} e^{(1-r^2)/2} $$
$$ \delta v = \frac{S}{2\pi} \frac{x-x_c}{R} e^{(1-r^2)/2} $$
$$ T = 1 - \frac{(\gamma-1)S^2}{8\gamma\pi^2} e^{1-r^2} $$
$$ \rho = T^{\frac{1}{\gamma-1}}, \quad p = \rho^\gamma $$

In [ ]:
import sys
import os
import numpy as np
try:
    import warp as wp
except ImportError:
    print("Error: warp-lang not found. Please install it with 'pip install warp-lang' or use the .venv environment.")
import matplotlib.pyplot as plt

# Add project root to path (robust discovery)
def find_project_root():
    # Check if we already set it in Colab cell
    if 'repo_path' in globals():
        return repo_path
        
    curr = os.getcwd()
    # Search upwards for solver.py to find the root
    while curr != os.path.dirname(curr):
        if os.path.exists(os.path.join(curr, 'solver.py')):
            return curr
        curr = os.path.dirname(curr)
    # Fallback to relative path
    return os.path.abspath("../../../")

project_root = find_project_root()
if project_root not in sys.path:
    sys.path.append(project_root)
print(f"Project root added to sys.path: {project_root}")

try:
    from solver import PazuzuSolver
    from src.kernels.initial_conditions import init_isentropic_vortex
    print("Successfully imported Solver modules.")
except ImportError as e:
    print(f"Import failed: {e}")
    print("Ensure you are running this notebook from within the project structure.")

In [ ]:
config_path = os.path.join(project_root, "tests/verification/vortex_2D/vortex.yaml")
if not os.path.exists(config_path):
    # Fallback for local run if already in the directory
    config_path = "vortex.yaml"

from src.core.config import PazuzuConfig
config = PazuzuConfig.from_yaml(config_path)

# --- Experiment with Overrides here! ---
config = config.override({
    "simulation.device": "cuda" if wp.is_cuda_available() else "cpu",
    "numerics.polynomial_order": 2,      # Degree of polynomials (N=2 -> 3rd order)
    "amr.initial_depth": 5,             # Grid resolution (2^depth x 2^depth blocks)
    "numerics.cfl": 0.1,                # Time step stability factor
    "simulation.t_final": 10.0,          # Simulation duration
    
    # Domain Bounds
    "mesh.x_min": -5.0,
    "mesh.x_max": 5.0,
    "mesh.y_min": -5.0,
    "mesh.y_max": 5.0,
    
    # Vortex Parameters
    "initial_condition.params.beta": 5.0,   # Strength of the vortex
    "initial_condition.params.radius": 1.0, # Radius of the vortex
    
    # I/O Settings
    "io.output_dir": "output_vortex",
    "io.write_interval": 100                # Save every 100 steps
})
# ---------------------------------------

solver = PazuzuSolver(config)
print(f'Initialized solver with N={solver.basis.N} (Order {solver.basis.N+1})')
print(f"Device: {solver.device}, Polynomial Order: {solver.config.numerics.polynomial_order}")

In [ ]:
# Visualize Initial Density and Velocity
q = solver.state.q.numpy()
rho = q[:, :, 0]
u = q[:, :, 1] / rho
v = q[:, :, 2] / rho
vel_mag = np.sqrt(u**2 + v**2)

x = solver.state.x.numpy().flatten()
y = solver.state.y.numpy().flatten()
rho_flat = rho.flatten()
vel_flat = vel_mag.flatten()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

im1 = ax1.tripcolor(x, y, rho_flat, cmap='viridis', shading='gouraud')
plt.colorbar(im1, ax=ax1, label='Density')
ax1.set_title('Initial Density')
ax1.set_aspect('equal')

im2 = ax2.tripcolor(x, y, vel_flat, cmap='magma', shading='gouraud')
plt.colorbar(im2, ax=ax2, label='Velocity Magnitude')
ax2.set_title('Initial Velocity Magnitude')
ax2.set_aspect('equal')

plt.tight_layout()
plt.show()

In [ ]:
print(f"Running simulation to t={solver.config.simulation.t_final}...")
solver.run()
print("Done.")

In [ ]:
# Compute Errors using shared logic
from tests.verification.vortex_2D.analyze_vortex import compute_errors, get_expected_error

L2_error, Linf_error, diff_rho = compute_errors(solver)
beta = float(solver.config.initial_condition.params.get('beta', 5.0))
expected_error = get_expected_error(solver.basis.N, solver.config.amr.initial_depth, beta)

print(f'L2 Error:       {L2_error:.6e}')
print(f'Linf Error:     {Linf_error:.6e}')
print(f'Expected Error: {expected_error:.6e}')
print(f'Steps: {solver.state.step}, Time: {solver.state.t:.4f}')

## Dynamic Visualization
Animate the vortex convection from the saved HDF5 data.

In [ ]:
import h5py
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# The solver currently hardcodes the filename to 'results.h5'
h5_path = os.path.join(solver.config.io.output_dir, "results.h5")

if os.path.exists(h5_path):
    with h5py.File(h5_path, 'r') as f:
        pts = f['mesh/points'][:]
        px = pts[:, 0]
        py = pts[:, 1]
        
        steps = sorted(f['data'].keys(), key=lambda x: int(x.split('_')[1]))
        
        fig, ax = plt.subplots(figsize=(6, 5))
        # Initial plot using tripcolor
        u0 = f[f'data/{steps[0]}/u'][:]
        v0 = f[f'data/{steps[0]}/v'][:]
        vel0 = np.sqrt(u0**2 + v0**2)
        tc = ax.tripcolor(px, py, vel0, cmap='magma', shading='gouraud')
        fig.colorbar(tc, label='Velocity Magnitude')
        ax.set_aspect('equal')
        title = ax.set_title(f"Velocity at t={f[f'data/{steps[0]}'].attrs['time']:.3f}")

        def update(step_name):
            u = f[f'data/{step_name}/u'][:]
            v = f[f'data/{step_name}/v'][:]
            vel = np.sqrt(u**2 + v**2)
            tc.set_array(vel)
            title.set_text(f"Velocity at t={f[f'data/{step_name}'].attrs['time']:.3f}")
            return tc, title

        anim = FuncAnimation(fig, update, frames=steps, interval=100, blit=True)
        plt.close() # Prevent static plot
        display(HTML(anim.to_jshtml()))
else:
    print(f"HDF5 file not found at {h5_path}. Ensure the simulation ran and saved steps.")

## Results Analysis

The simulation accuracy is evaluated using two primary error norms for the density field:

1.  **$L_2$ Error (RMS Error):** Represents the integrated (global) error across the domain. In High-Order methods like Flux Reconstruction, this error should decrease as $O(h^{N+1})$ where $N$ is the polynomial degree.
2.  **$L_{inf}$ Error (Maximum Error):** Represents the largest pointwise discrepancy. This is a sensitive measure of local oscillations or resolution issues near the vortex core.

### Final Error Metrics

In [ ]:
print(f"Final simulation time: t = {solver.state.t:.4f}")
print(f"Polynomial Order (N):  {solver.basis.N}")
print(f"Initial Depth:        {solver.config.amr.initial_depth}")
print("-" * 30)
print(f"L2 Error (Density):   {L2_error:.6e}")
print(f"Linf Error (Density): {Linf_error:.6e}")

In [ ]:
# Visualize Pointwise Error
plt.figure(figsize=(6,5))
plt.tripcolor(x, y, np.abs(diff_rho).flatten(), cmap='inferno', shading='gouraud')
plt.colorbar(label='Absolute Error (Density)')
plt.title(f'Spatial Error Distribution (t={solver.state.t:.2f})')
plt.axis('equal')
plt.show()